In [1]:
pip install ultralytics timm opencv-python torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.2 MB/s eta 0:00:00


In [2]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nirmalsankalana/plantdoc-dataset
!unzip plantdoc-dataset.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/nirmalsankalana/plantdoc-dataset
License(s): CC0-1.0
 98% 877M/896M [00:17<00:00, 62.0MB/s]
100% 896M/896M [00:17<00:00, 54.8MB/s]
Archive:  plantdoc-dataset.zip
  inflating: file_renamer.py         
  inflating: folder_renamer.py       
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_1.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_10.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_2.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_3.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_4.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_5.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_6.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_7.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_8.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_9.jpg  
  inflating: test/Apple_leaf/test_Apple leaf_1.jpg  

In [ ]:
import os
import cv2
import numpy as np
from ultralytics import SAM
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"

INPUT_ROOT = "train"
OUTPUT_ROOT = "segmented_dataset_new_base"

selected_classes = [
    "Bell_pepper_leaf",
    "Bell_pepper_leaf_spot",
    "Potato_leaf_early_blight",
    "Potato_leaf_late_blight"
]

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Loading SAM2-Base model...")
sam_model = SAM("sam2_b.pt")
sam_model.to(DEVICE)

def segment_tight_crop(image):

    h, w = image.shape[:2]

    results = sam_model(image, device=DEVICE)

    if results[0].masks is None:
        return image

    masks = results[0].masks.data.cpu().numpy()

    if len(masks) == 0:
        return image

    masks = [m for m in masks if m.sum() > 0.02 * h * w]
    if len(masks) == 0:
        return image

    mask = max(masks, key=lambda x: x.sum())

    ys, xs = np.where(mask > 0)
    y1, y2 = ys.min(), ys.max()
    x1, x2 = xs.min(), xs.max()

    pad_y = int((y2 - y1) * 0.05)
    pad_x = int((x2 - x1) * 0.05)

    y1 = max(0, y1 - pad_y)
    y2 = min(h, y2 + pad_y)
    x1 = max(0, x1 - pad_x)
    x2 = min(w, x2 + pad_x)

    return image[y1:y2, x1:x2]

print("Starting segmentation")

for cls in selected_classes:

    input_folder = os.path.join(INPUT_ROOT, cls)
    output_folder = os.path.join(OUTPUT_ROOT, cls)
    os.makedirs(output_folder, exist_ok=True)

    images = os.listdir(input_folder)
    print(f"\nProcessing class: {cls} | {len(images)} images")

    for i, img_name in enumerate(images):

        img_path = os.path.join(input_folder, img_name)
        image = cv2.imread(img_path)

        cropped = segment_tight_crop(image)

        save_path = os.path.join(output_folder, img_name)
        cv2.imwrite(save_path, cropped)

        if i % 10 == 0:
            print(f"{i}/{len(images)} done")

print("\nSegmentation completed")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading SAM2-Base model...
Starting segmentation

Processing class: Bell_pepper_leaf | 34 images

0: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 1 19, 1 20, 1 21, 1 22, 1 23, 1 24, 1 25, 1 26, 1 27, 1 28, 1 29, 1 30, 1 31, 1 32, 1 33, 1 34, 1 35, 1 36, 1 37, 1 38, 1 39, 1 40, 1 41, 1 42, 1 43, 1 44, 1 45, 1 46, 1 47, 1 48, 1 49, 1 50, 1 51, 1 52, 1 53, 1 54, 1 55, 1 56, 1 57, 1 58, 1 59, 1 60, 9347.0ms
Speed: 79.9ms preprocess, 9347.0ms inference, 7.7ms postprocess per image at shape (1, 3, 1024, 1024)
0/34 done

0: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 7772.1ms
Speed: 8.0

In [4]:
import shutil
from google.colab import files

shutil.make_archive('segmented_dataset_new_base', 'zip', 'segmented_dataset_new_base')
files.download('segmented_dataset_new_base.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>